In [ ]:
import os.path
import torch
from pathlib import Path

from pyro_cases.run import vae_dict

In [2]:
def drop_not_converge_cases(data):
    new_favi_test_dict_list = []
    new_elbo_test_dict_list = []
    num_favi_not_converge = 0
    num_elbo_not_converge = 0
    for td in data:
        if "favi_cant_converge" in td["favi_test_dict_list"]:
            num_favi_not_converge += 1
        else:
            new_favi_test_dict_list.append(td["favi_test_dict_list"])
        
        if "elbo_cant_converge" in td["elbo_test_dict_list"]:
            num_elbo_not_converge += 1
        else:
            new_elbo_test_dict_list.append(td["elbo_test_dict_list"])
    return new_favi_test_dict_list, num_favi_not_converge, new_elbo_test_dict_list, num_elbo_not_converge

In [3]:
def load_data(data_path):
    data = torch.load(data_path, map_location=torch.device("cpu"))
    model_num = len(data)
    task_name = data[0]["task"]
    obs_num = 10
    theta_dim = vae_dict[task_name].theta_dim

    favi_est_mu = torch.full((model_num, obs_num, theta_dim), torch.nan)
    favi_est_sigma2 = torch.full((model_num, obs_num, theta_dim), torch.nan)
    elbo_est_mu = torch.full((model_num, obs_num, theta_dim), torch.nan)
    elbo_est_sigma2 = torch.full((model_num, obs_num, theta_dim), torch.nan)

    favi_test_dict_list, num_favi_not_converge, elbo_test_dict_list, num_elbo_not_converge = drop_not_converge_cases(data)
    assert len(favi_test_dict_list) > 0
    for i, tdl in enumerate(favi_test_dict_list):
        for j, td in enumerate(tdl):
            favi_est_mu[i, j] = td["est_mu"]
            favi_est_sigma2[i, j] = td["est_sigma2"]
    for i, tdl in enumerate(elbo_test_dict_list):
        for j, td in enumerate(tdl):
            elbo_est_mu[i, j] = td["est_mu"]
            elbo_est_sigma2[i, j] = td["est_sigma2"]

    true_theta = torch.zeros(obs_num, theta_dim)
    for obs_index in range(obs_num):
        true_theta[obs_index] = favi_test_dict_list[0][obs_index]["true_theta"]
    return {
        "task_name": task_name,
        "obs_num": obs_num,
        "theta_dim": theta_dim,
        "favi_est_mu": favi_est_mu,
        "favi_est_sigma2": favi_est_sigma2,
        "num_favi_not_converge": num_favi_not_converge,
        "elbo_est_mu": elbo_est_mu,
        "elbo_est_sigma2": elbo_est_sigma2,
        "num_elbo_not_converge": num_elbo_not_converge,
        "true_theta": true_theta,
    }

In [4]:
def nanstd(tensor, dim=None, keepdim=False):
    tensor_mean = tensor.nanmean(dim=dim, keepdim=True)
    output = (tensor - tensor_mean).square().nanmean(dim=dim, keepdim=keepdim)
    return output.sqrt()

In [5]:
def test_data_std(data_path):
    if not os.path.isfile(data_path):
        print(f"skip {data_path}")
        return None
    
    data_dict = load_data(data_path)
    favi_est_mu_std = nanstd(data_dict["favi_est_mu"], dim=0).mean()
    favi_est_sigma2_std = nanstd(data_dict["favi_est_sigma2"], dim=0).mean()
    elbo_est_mu_std = nanstd(data_dict["elbo_est_mu"], dim=0).mean()
    elbo_est_sigma2_std = nanstd(data_dict["elbo_est_sigma2"], dim=0).mean()

    return {
        "favi_est_mu_std": favi_est_mu_std, 
        "favi_est_sigma2_std": favi_est_sigma2_std, 
        "num_favi_not_converge": data_dict["num_favi_not_converge"],
        "elbo_est_mu_std": elbo_est_mu_std, 
        "elbo_est_sigma2_std": elbo_est_sigma2_std,
        "num_elbo_not_converge": data_dict["num_elbo_not_converge"],
    }

In [ ]:
# use vsbc or/and mcmc

data_path = Path("/data/scratch/pduan/gcvi_03-25_output")
tasks = list(vae_dict.keys())
lr_schedulers = ["cosine_annealing"]
network_widths = [1024]
for t in tasks:
    for nw in network_widths:
        for lr_s in lr_schedulers:
            tag = f"pyro_t_{t}_lr_{lr_s}_nw_{nw}_mn_100.pt"
            data_dict = test_data_std(data_path / tag)
            print("=" * 100)
            print(f"task: {t}")
            for k, v in data_dict.items():
                if "num" not in k:
                    print(f"\t {k}: {v:.2e}")
                else:
                    print(f"\t {k}: {v:d}")

task: gaussian_linear
	 favi_est_mu_std: 3.77e-03
	 favi_est_sigma2_std: 3.42e-04
	 num_favi_not_converge: 0
	 elbo_est_mu_std: 3.83e-03
	 elbo_est_sigma2_std: 3.69e-04
	 num_elbo_not_converge: 0
task: gaussian_linear_uniform
	 favi_est_mu_std: 1.33e-02
	 favi_est_sigma2_std: 8.04e-04
	 num_favi_not_converge: 0
	 elbo_est_mu_std: 1.47e-02
	 elbo_est_sigma2_std: 1.00e-03
	 num_elbo_not_converge: 0
task: slcp
	 favi_est_mu_std: 4.63e-03
	 favi_est_sigma2_std: 4.78e-03
	 num_favi_not_converge: 0
	 elbo_est_mu_std: 2.96e-01
	 elbo_est_sigma2_std: 4.14e-01
	 num_elbo_not_converge: 0
task: slcp_distractors
	 favi_est_mu_std: 2.05e-02
	 favi_est_sigma2_std: 3.35e-02
	 num_favi_not_converge: 0
	 elbo_est_mu_std: 4.67e-01
	 elbo_est_sigma2_std: 4.70e-01
	 num_elbo_not_converge: 0
task: bernoulli_glm_raw
	 favi_est_mu_std: 4.95e-02
	 favi_est_sigma2_std: 5.49e-03
	 num_favi_not_converge: 0
	 elbo_est_mu_std: 3.67e-02
	 elbo_est_sigma2_std: 2.44e-03
	 num_elbo_not_converge: 0
task: bernoulli_glm
